# Data Cleaning — SAP News Intelligence
Cleans `clean_data.json` (56 articles, 9 columns) and saves a validated `clean_data.json` ready for chunking and embedding.

**Steps covered:**
1. Load & inspect raw data
2. Normalize unicode / whitespace
3. Parse & sort dates
4. Reconstruct missing URLs
5. Deduplicate
6. Drop short/empty rows
7. Add `data_source_type` column
8. Save & validate


## 1. Imports & configuration

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR      = Path("../data")          # adjust if needed
RAW_PATH      = DATA_DIR / "clean_data.json"
OUT_PATH      = DATA_DIR / "clean_data.json"   # overwrite in-place

MIN_TEXT_LEN  = 50    # drop articles shorter than this (failed scrapes)


## 2. Load raw data

In [ ]:
df = pd.read_json(RAW_PATH)
print(f"Shape:   {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)


In [ ]:
print("─── Null counts ───")
print(df.isnull().sum())
print(f"\nDtypes:\n{df.dtypes}")


## 3. Normalize unicode & whitespace

Raw scrapes often contain:
- Non-breaking spaces (`\u00a0`), zero-width chars (`\u200b`), BOM (`\ufeff`)
- Stray HTML tags left by the scraper
- Consecutive spaces / mixed newlines


In [ ]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Canonical unicode form (converts curly quotes, ligatures, etc.)
    text = unicodedata.normalize("NFKC", text)
    # Remove invisible / control characters
    text = re.sub(r"[\u200b\u00a0\u200e\u200f\ufeff\u2028\u2029]", " ", text)
    # Strip stray HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    # Collapse all whitespace to single space
    text = re.sub(r"\s+", " ", text)
    return text.strip()


before_sample = df.loc[0, "clean_text"][:120]

df["title"]      = df["title"].map(normalize_text)
df["clean_text"] = df["clean_text"].map(normalize_text)
df["source"]     = df["source"].str.strip()
df["clean_length"] = df["clean_text"].str.len()

after_sample = df.loc[0, "clean_text"][:120]
print("Before:", repr(before_sample))
print("After: ", repr(after_sample))


## 4. Parse & sort dates

- SAP News articles have proper ISO 8601 timestamps
- 26 GNews articles have `null` dates (no pub date in the feed) — kept as `NaT`, not dropped


In [ ]:
df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)

n_null = int(df["published"].isna().sum())
print(f"Rows with no date (NaT): {n_null}  — sources: {df.loc[df['published'].isna(), 'source'].value_counts().to_dict()}")
print(f"Date range (non-null):   {df['published'].min()}  →  {df['published'].max()}")

# Sort chronologically; rows with no date go to the end
df = df.sort_values("published", na_position="last").reset_index(drop=True)
print(f"\nSorted: {df['published'].is_monotonic_increasing} (ignoring NaT tail)")


## 5. Reconstruct missing URLs

30 SAP News articles had no URL in the raw data.  
We reconstruct a canonical slug from the title (`https://news.sap.com/2026/<slug>/`).


In [ ]:
def make_sap_slug(title: str) -> str:
    slug = title.lower()
    slug = re.sub(r"[^a-z0-9\s-]", "", slug)
    slug = re.sub(r"\s+", "-", slug).strip("-")
    return f"https://news.sap.com/2026/{slug}/"


mask_no_url = df["url"].isna() | (df["url"].astype(str).str.strip() == "")
print(f"Rows missing URL: {mask_no_url.sum()}")

df.loc[mask_no_url & (df["source"] == "SAP News"), "url"] = (
    df.loc[mask_no_url & (df["source"] == "SAP News"), "title"]
    .map(make_sap_slug)
)

print(f"Still missing after reconstruction: {df['url'].isna().sum()}")
print("\nExample reconstructed URL:")
print(df[df["source"] == "SAP News"]["url"].iloc[0])


## 6. Deduplicate

In [ ]:
before = len(df)

df = df.drop_duplicates(subset=["url"])
df = df.drop_duplicates(subset=["title", "source"])

after = len(df)
print(f"Removed {before - after} duplicate rows ({before} → {after})")


## 7. Drop short / empty rows

Any article with `clean_text` shorter than `50` chars is likely a failed scrape.

In [ ]:
short_mask = df["clean_text"].str.len() < MIN_TEXT_LEN
print(f"Rows below minimum length ({MIN_TEXT_LEN} chars): {short_mask.sum()}")
df = df[~short_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df)}")


## 8. Add `data_source_type` column

Distinguishes SAP-owned content from third-party press coverage — useful for filtering in downstream analysis.

In [ ]:
df["data_source_type"] = df["source"].apply(
    lambda s: "owned" if s == "SAP News" else "third_party"
)

print(df["data_source_type"].value_counts().to_string())


## 9. Summary report

In [ ]:
print("═" * 45)
print("  FINAL DATA CLEANING SUMMARY")
print("═" * 45)
print(f"  Total rows          : {len(df)}")
print(f"  Columns             : {len(df.columns)}")
print(f"  Missing dates (NaT) : {df['published'].isna().sum()}  (GNews — no date in feed)")
print(f"  Missing URLs        : {df['url'].isna().sum()}")
print(f"  Duplicate rows      : {df.duplicated().sum()}")
print(f"  Min clean_length    : {df['clean_length'].min()}")
print(f"  Max clean_length    : {df['clean_length'].max()}")
print(f"  Mean clean_length   : {df['clean_length'].mean():.0f}")
print(f"  Owned (SAP News)    : {(df['data_source_type']=='owned').sum()}")
print(f"  Third-party         : {(df['data_source_type']=='third_party').sum()}")
print("═" * 45)

df.head(3)


## 10. Save cleaned data

In [ ]:
df.to_json(OUT_PATH, orient="records", indent=4, date_format="iso")
print(f"Saved → {OUT_PATH}  ({len(df)} rows, {len(df.columns)} columns)")
